# Laboratorio 11.1 — Observabilidad con LangSmith

Instrumentaremos el workflow LangGraph construido en laboratorios anteriores. Las trazas y recursos remotos quedan desactivados hasta habilitarlos explícitamente.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Instrumentar un workflow previo con trazas y runs jerárquicos.
- Preparar datasets y evaluadores para experimentos versionados.
- Inspeccionar latencia, errores y metadatos de tokens cuando existan.
- Habilitar LangSmith solo con consentimiento, clave y configuración explícita.

## 2. Conceptos

Una traza agrupa la operación; sus runs representan componentes, entradas, salidas, latencia y errores. Datasets guardan ejemplos de evaluación; experimentos comparan una versión del sistema sobre esos ejemplos, y evaluadores calculan métricas. Trazas pueden contener información sensible: no las envíes sin revisar datos, retención y permisos. Tokens y costo solo se informan si el modelo los entrega y existe una tarifa confiable.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Dataset local] --> B[Workflow LangGraph]
    B --> C[Trace y runs]
    C --> D[Latencia, errores y tokens]
    B --> E[Resultados locales]
    D --> F[Evaluador]
    E --> F
    F -. opcional .-> G[Dataset y experimento LangSmith]
```

## 4. Preparación del entorno

Instrumentamos el grafo del laboratorio 10.2 y leemos configuración sin imprimir secretos. La ruta normal no crea un cliente LangSmith ni envía trazas.

In [ ]:
from datetime import datetime, timezone
from time import perf_counter

import pandas as pd

from llm_engineering.environment import preparar_entorno
from llm_engineering.agents import build_workflow
from llm_engineering.observability import (
    langsmith_tracing, load_langsmith_settings, traceable_if_enabled,
)

raiz_curso = preparar_entorno()
settings_tracing = load_langsmith_settings()
print({
    'tracing_enabled': settings_tracing.enabled,
    'project': settings_tracing.project_name,
    'api_key_configured': settings_tracing.api_key_configured,
})

## 5. Definición del problema

Mediremos el mismo workflow FAQ que enruta a recuperación, herramienta o respuesta directa. El conjunto local contiene tres consultas con rutas esperadas; sirve como smoke dataset para analizar runs, no como benchmark de calidad semántica.

## 6. Línea base

Ejecutamos el grafo sin trazas remotas y medimos tiempo de pared. Esta línea base permite diferenciar latencia local de una integración de observabilidad y conserva el resultado si LangSmith no está configurado.

In [ ]:
dataset_local = pd.DataFrame([
    {'id': 'obs-1', 'request': 'devolucion 30 dias', 'expected_route': 'rag'},
    {'id': 'obs-2', 'request': 'calcula 5 + 4', 'expected_route': 'tool'},
    {'id': 'obs-3', 'request': 'hola', 'expected_route': 'direct'},
])
workflow = build_workflow()

def ejecutar_sin_traza(request: str) -> dict[str, object]:
    """Invoca el workflow anterior y retorna su salida validada."""
    resultado = workflow.invoke({'request': request})
    return resultado['validated_output']

print('Ejemplos locales:', len(dataset_local))

## 7. Implementación

Decoramos una función que ya ejecuta LangGraph. Con `LANGSMITH_TRACING=false` la utilidad retorna la función original; al activarla, el SDK crea runs. Cada llamada recibe solo una solicitud conocida y las trazas respetan la configuración del entorno.

In [ ]:
@traceable_if_enabled(name='workflow_faq')
def ejecutar_workflow(request: str) -> dict[str, object]:
    """Instrumenta la invocacion del workflow de LangGraph."""
    resultado = workflow.invoke({'request': request})
    return resultado['validated_output']

filas = []
for ejemplo in dataset_local.to_dict(orient='records'):
    inicio = perf_counter()
    try:
        with langsmith_tracing():
            salida = ejecutar_workflow(ejemplo['request'])
        error = None
    except Exception as excepcion:
        salida = {'route': None, 'status': 'error', 'source_ids': []}
        error = type(excepcion).__name__
    filas.append({
        **ejemplo,
        'route': salida.get('route'),
        'schema_valid': salida.get('status') in {'ok', 'error'},
        'source_ids': salida.get('source_ids', []),
        'latency_seconds': perf_counter() - inicio,
        'input_tokens': None,
        'output_tokens': None,
        'estimated_cost': None,
        'error_type': error,
    })
resultados = pd.DataFrame(filas)
display(resultados)

## 8. Experimentos

Comparamos rutas y resultados del sistema con el dataset fijo. Si LangSmith se activa, el bloque remoto crea un dataset único y ejecuta como máximo tres ejemplos; no envía claves al notebook ni llama a un proveedor de modelos.

In [ ]:
resultados['route_match'] = resultados['route'] == resultados['expected_route']
display(resultados[['id', 'expected_route', 'route', 'route_match', 'latency_seconds', 'error_type']])
resultados.to_csv(raiz_curso / 'outputs' / '11_01_observability_runs.csv', index=False)

PUBLICAR_EN_LANGSMITH = False
if PUBLICAR_EN_LANGSMITH:
    if not settings_tracing.enabled or not settings_tracing.api_key_configured:
        raise ValueError('Configura LANGSMITH_TRACING=true y LANGSMITH_API_KEY antes de publicar.')
    from langsmith import Client, evaluate

    client = Client()
    nombre_dataset = f"curso-llm-observability-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
    dataset = client.create_dataset(dataset_name=nombre_dataset, description='Smoke dataset local para el workflow FAQ.')
    client.create_examples(
        inputs=[{'request': fila['request']} for fila in dataset_local.to_dict(orient='records')],
        outputs=[{'expected_route': fila['expected_route']} for fila in dataset_local.to_dict(orient='records')],
        dataset_id=dataset.id,
)

    def evaluar_ruta(run, example) -> dict[str, object]:
        """Evalua exactitud de ruta contra la referencia del dataset."""
        coincide = run.outputs.get('route') == example.outputs.get('expected_route')
        return {'key': 'route_exact_match', 'score': float(coincide)}

    experimento = evaluate(
        ejecutar_workflow,
        data=dataset.name,
        evaluators=[evaluar_ruta],
        experiment_prefix='workflow-faq',
)
    print('Dataset y experimento creados:', dataset.name, experimento)

## 9. Evaluación

Exactitud de ruta y latencia se calculan localmente; el dataset pequeño no mide calidad de respuesta. En la ruta actual no existe llamada a LLM, por eso tokens y costo son `None`, no cero. LangSmith agrega inspección de runs y evaluadores si se habilita explícitamente.

In [ ]:
metricas_locales = pd.DataFrame([{
    'route_accuracy': float(resultados['route_match'].mean()),
    'failure_rate': float(resultados['error_type'].notna().mean()),
    'latency_mean_seconds': float(resultados['latency_seconds'].mean()),
    'input_tokens_available': bool(resultados['input_tokens'].notna().any()),
    'output_tokens_available': bool(resultados['output_tokens'].notna().any()),
    'cost_available': bool(resultados['estimated_cost'].notna().any()),
}])
display(metricas_locales)
metricas_locales.to_csv(raiz_curso / 'outputs' / '11_01_observability_metrics.csv', index=False)

## 10. Discusión

Una traza puede registrar contenido confidencial, y compartirla con un servicio remoto requiere consentimiento. Inspecciona inputs/outputs, retención y acceso antes de activar. Los tokens dependen de metadatos del proveedor; estima costo solo con tarifas versionadas y valores verificables.

## 11. Ejercicios

1. Añade una salida inválida al dataset y clasifica su error.
2. Registra latencia por ruta y compara distribución, no solo promedio.
3. Habilita LangSmith con un dataset separado y evalúa una consulta local sin datos sensibles.
4. Inspecciona un run y marca qué campos podrían revelar información privada.

## 12. Desafío

Instrumenta el agente manual y el workflow RAG con una misma evaluación. Compara runs, errores, latencia, uso de tokens si existe y resultados de evaluadores humanos frente a automáticos; documenta cómo repetir el experimento.

## 13. Ideas clave

- Observabilidad debe instrumentar sistemas que ya se construyeron.
- Traces, runs, datasets, experimentos y evaluadores responden preguntas distintas.
- Integración remota es optativa y puede transferir datos.
- No inventes tokens ni costos cuando el runtime no los informa.